# MoCo v3 (ViT-S/16) Pretraining — Jupyter 서버 (RTX A5000 24GB)

> **최종 모델(ViT-S/8)은 `server_train_universal.ipynb` 사용.** 이 노트북은 ViT-S/16 단일 학습용.

## 사전 준비 (최초 1회 — JupyterLab Terminal)
```bash
git clone -b final-jupyter <repo-url> && cd <repo>
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt && pip install -e .
python -m ipykernel install --user --name ssl --display-name "Python (ssl)"
```

## 권장 실행 방식
장시간(수십 시간) 학습은 노트북 셀 대신 **Terminal + nohup** 권장
(커널 재시작 시 학습이 중단되고 셀 출력이 유실됨):
```bash
nohup python -u scripts/train_mocov3.py > logs/train_mocov3.out 2>&1 &
tail -f logs/train_mocov3.out
```
노트북에서 돌려야 하면 아래 Cell 2 사용 — 서버/프로세스 중단 시 Cell 2 재실행으로 마지막 체크포인트에서 자동 resume.

In [ ]:
# Cell 1 — 작업 디렉토리 = 레포 루트 + GPU 확인
# (서버에는 Drive 마운트/심링크 불필요 — data/outputs/logs/features는 레포 루트에 영구 보존)
import os, torch
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
print('Working dir:', os.getcwd())

assert torch.cuda.is_available(), 'GPU 사용 불가 — CUDA 환경(드라이버/torch cuda wheel)을 확인하세요.'
print(f'GPU : {torch.cuda.get_device_name(0)}')   # 기대값: NVIDIA RTX A5000
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'BF16: {torch.cuda.is_bf16_supported()}')  # A5000(Ampere) → True 필수

In [ ]:
# Cell 2 — 학습 시작 (자동 resume + 실시간 로그 출력)
import glob, re, subprocess

OUT_DIR = 'outputs/mocov3_vits_seed42'

# 마지막 체크포인트 자동 탐색 — epoch 번호 정수 정렬 (문자열 정렬은 ep100+에서 깨짐)
def ep_num(p):
    return int(re.search(r'ckpt_ep(\d+)\.pth', p).group(1))
ckpts = sorted(glob.glob(f'{OUT_DIR}/ckpt_ep*.pth'), key=ep_num)

if ckpts:
    print(f'Resume: {ckpts[-1]}')
    resume_args = ['--resume', ckpts[-1]]
else:
    print('처음부터 학습 시작')
    resume_args = []

# 서버에서는 num_workers / save_every 모두 config 값 사용 (override 불필요)
cmd = ['python3', '-u', 'scripts/train_mocov3.py'] + resume_args

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n학습 종료 (exit code: {proc.returncode})')

In [ ]:
# Cell 3 — 학습 상태 확인 (Cell 2 실행 중에는 실행 불가 — Terminal에서 tail 사용)
!echo '=== 최근 로그 ==='
!tail -n 20 logs/mocov3_vits_seed42.log
!echo ''
!echo '=== 저장된 체크포인트 ==='
!ls -lh outputs/mocov3_vits_seed42/
!echo ''
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv